##### Imports

In [ ]:
import pandas as pd
import numpy as np
import csv
import json

from pathlib import Path
from enum import Enum

In [ ]:
try:
    # Works when running a .py file
    ROOT = Path(__file__).parent.resolve()
except NameError:
    # Works in notebooks / interactive
    ROOT = Path.cwd().resolve()

##### Prepare Dataset

In [ ]:
DATA_DIR = "Data Directory"
PROCESSED_DATA_DIR = "processed_data/"
ADD_CAT = True # Whether to add category token at the beginning
SEED = 42 # Set a random seed for reproducibility

In [ ]:
class Cols(str, Enum): # Column names in the dataset/dataframe
    Record_Number = "Record Number"
    Category = "Category"
    Title = "Title"
    Token = "Token"
    Tag = "Tag"
    Token_Tag_List = "Token_Tag_List"

In [ ]:
df = pd.read_csv(
    DATA_DIR, 
    sep="\t",
    keep_default_na=False,
    na_values=None, 
    encoding="utf-8",
    header=0,)

In [ ]:
df.head()

In [ ]:
''' Group by Record Number and create a list of (Token, BIO-Tag) tuples for each title '''
def get_token_tag_list(cur_df):
    token_tags = list(zip(cur_df[Cols.Token], cur_df[Cols.Tag]))
    cur_cat = cur_df[Cols.Category].iloc[0]

    combined_token_tags = []
    if ADD_CAT:
        combined_token_tags.append(("KAT_"+str(cur_cat), "O")) # Add category token at the beginning
    
    latest_tag = ""
    for i, (token, tag) in enumerate(token_tags):
        cur_token = token.strip()
        if tag == "O":
            latest_tag = tag
            tag = "O"
        elif tag == "":
            tag = "I-"+latest_tag
            if tag == "I-O":
                print("Warning: Incorrect I-O tag in record number 1827, replacing with B-Kompatibles_Fahrzeug_Modell")
                tag = "B-Kompatibles_Fahrzeug_Modell"
            elif tag == "I-":
                print("Error!!!!!!!!!!!!")
        else:
            latest_tag = tag
            tag = "B-"+tag
            
        combined_token_tags.append((cur_token, tag))

    return pd.Series({
            Cols.Record_Number: cur_df[Cols.Record_Number].iloc[0],  # keep one record number
            Cols.Category: cur_df[Cols.Category].iloc[0],  # keep one category
            Cols.Title: cur_df[Cols.Title].iloc[0],  # keep one title
            Cols.Token_Tag_List: combined_token_tags # list of (Token, Tag) tuples
          })


In [ ]:
def save_data(cur_df, filename):
    with open(filename,"w", encoding="utf-8") as f:
        for index, row in cur_df.iterrows():
            cur_data = {}
            cur_data["tokens"] = [token for token, tag in row[Cols.Token_Tag_List]]
            cur_data["tags"] = [tag for token, tag in row[Cols.Token_Tag_List]]
            f.write(json.dumps(cur_data, ensure_ascii=False)+"\n")

In [ ]:
df_token_tag = df.groupby(Cols.Record_Number)[df.columns].apply(get_token_tag_list)
save_data(df_token_tag, "ner_data/all_data.jsonl")